In [18]:
# import pandas as pd
import pandas as pd
import numpy as np

Loading the Data set 

In [19]:
sessions = pd.read_csv("datasets/stadiochoice/table1_trial_viewer_behaviour.csv")
billing = pd.read_csv("datasets/stadiochoice/table2_subscription_billing.csv")
outcome = pd.read_csv("datasets/stadiochoice/table3_conversion_outcome.csv")
demographics = pd.read_csv("datasets/stadiochoice/table4_trial_user_demographics.csv")


In [20]:
# Inspect the sessions  data set
print("sessions shape:", sessions.shape)
print("sessions duplicate count:", sessions.duplicated().sum())
print("sessions head:")
display(sessions.head())

sessions shape: (123470, 12)
sessions duplicate count: 369
sessions head:


,account_id,session_id,session_start,trial_start_date,main_device,genre_watched_per_session,search_query_text,session_watch_time,total_watch_time,genres_watched_list,searches_made,no_results_searches
0,ACC-4000000,SESS-000000,2024-11-28 13:15,2024-11-20,Smart TV,Reality,big brother,48.9,1826.0,Reality,46,7
1,ACC-4000000,SESS-000001,2024-12-01 14:36,2024-11-20,Smart TV,Reality,NaN,33.8,1826.0,Reality,46,7
2,ACC-4000000,SESS-000002,2024-11-24 02:09,2024-11-20,Smart TV,Reality,NaN,81.3,1826.0,Reality,46,7
3,ACC-4000000,SESS-000003,2024-11-27 02:10,2024-11-20,Smart TV,Reality,NaN,48.4,1826.0,Reality,46,7
4,ACC-4000000,SESS-000004,2024-11-28 23:28,2024-11-20,Smart TV,Reality,reality show; cooking competition,35.1,1826.0,Reality,46,7


In [21]:
# Inspect the data set
print("billing shape:", billing.shape)
print("billing duplicate count:", billing.duplicated().sum())
print("billing head:")
display(billing.head())

billing shape: (18756, 6)
billing duplicate count: 56
billing head:


,account_id,payment_plan_filled_in,sign_up_platform,region,referral_or_promotion_code,trial_plan_chosen
0,ACC-4000000,True,Promotion Ad,KZN,True,Basic
1,ACC-4000003,True,App,GP,False,Premium
2,ACC-4000006,True,Website,WC,False,Standard
3,ACC-4000008,False,Website,GP,False,Premium
4,ACC-4000010,True,App,EC,False,Premium


In [22]:
# Inspect the data set
print("outcome shape:", outcome.shape)
print("outcome duplicate count:", outcome.duplicated().sum())
print("outcome head:")
display(outcome.head())

outcome shape: (18756, 4)
outcome duplicate count: 56
outcome head:


,account_id,converted_to_paid_subscription,date_of_conversion,early_churn
0,ACC-4000000,True,2024-12-02,False
1,ACC-4000003,True,2025-05-27,False
2,ACC-4000006,True,2025-02-13,False
3,ACC-4000008,True,2025-03-31,False
4,ACC-4000010,False,NaN,NaN


In [23]:
# Inspect the demographics data set
print("demographics:", demographics.shape)
print("demographics duplicate count:", demographics.duplicated().sum())
print("demographics head:")
display(demographics.head())

demographics: (18756, 3)
demographics duplicate count: 56
demographics head:


,account_id,age_band,stated_interests
0,ACC-4000000,25-34,"News, Comedy, Sport"
1,ACC-4000003,35-44,"Movies, News, Local content, Kids"
2,ACC-4000006,25-34,"Reality, Documentaries"
3,ACC-4000008,25-34,"Sport, News"
4,ACC-4000010,55+,"Reality, Comedy, Kids"


In [24]:
print("sessions unique accounts:", sessions['account_id'].nunique())
print("billing unique accounts:", billing['account_id'].nunique())
print("outcome unique accounts:", outcome['account_id'].nunique())
print("demographics unique accounts:", demographics['account_id'].nunique())

print("\nConversion counts:")
print(outcome['converted_to_paid_subscription'].value_counts())

print("\nConversion proportions (normalise=True):")
print(outcome['converted_to_paid_subscription'].value_counts(normalize=True))


sessions unique accounts: 18700
billing unique accounts: 18700
outcome unique accounts: 18700
demographics unique accounts: 18700

Conversion counts:
converted_to_paid_subscription
False    9912
True     8844
Name: count, dtype: int64

Conversion proportions (normalise=True):
converted_to_paid_subscription
False    0.528471
True     0.471529
Name: proportion, dtype: float64



# The loaded data show:
 **Duplicates:** There were 123 470 session rows which covered 18700 unquie accounts while the billing, outcome and demographics tables have 18756 rows. There were 369 duplicate rows in sessions and 56 in each of the other three. I will remove this once im in the preprocessing phase.
 
 **Classes and counts:**  The target column (converted to paid subscription) has two classes "true" for the users who converted to paid and False for the ones who didnt. So 9912 False and 8844 True 

 **Imbalance:** : The target split is around 47% True and 53% False which is pretty balanced which thne makes it easier to work with compared to SS2 where there was 11% positive cases.

 **Why the four separate tables:** Ive kept them separate for nor because the "sessions" records every single stream, so one user would have mutliple rows. Then the other three tables have one row per account. Having them seperate will help me clean before merging.
 

PreProcessing the Data set

In [25]:
# removing the duplicates from the 4 tables
print("sessions shape before:", sessions.shape)
sessions = sessions.drop_duplicates()
print("sessions shape after:", sessions.shape)

print("billing shape before:", billing.shape)
billing = billing.drop_duplicates()
print("billing shape after:", billing.shape)

print("outcome shape before:", outcome.shape)
outcome = outcome.drop_duplicates()
print("outcome shape after:", outcome.shape)

print("demographics shape before:", demographics.shape)
demographics = demographics.drop_duplicates()
print("demographics shape after:", demographics.shape)

# check the repeated sessionID values
print("Duplicate session_id count:", sessions['session_id'].duplicated().sum())

sessions shape before: (123470, 12)
sessions shape after: (123101, 12)
billing shape before: (18756, 6)
billing shape after: (18700, 6)
outcome shape before: (18756, 4)
outcome shape after: (18700, 4)
demographics shape before: (18756, 3)
demographics shape after: (18700, 3)
Duplicate session_id count: 0


### Firstly removed the duplicates

**What I removed:** I removed the 369 dupicate rows that were in the "sessions" table, 56 duplicate rows from the account tables for billing, outcome and demographics.
**Reason why I removed:** If there duplicates were not removed, there would be a high chance of double counting and not giving the true reflection in numbers like the total watch time which can lead to a data leakage by having the same account in the training and test data sets. 
**How I checked it worked:** After cleaning the data by removing, the billing, outcome and demographics all ended up on 18700 rows which then matched the 18700 unique accounts. I also checked for repeated session Ids and the results were then zero, which shows that there werent sessions show twice in the table. 

In [26]:
# checking the missing values in the tables in the tables 
print("sessions missing values:")
print(sessions.isna().sum())

print("\nbilling missing values:")
print(billing.isna().sum())

print("\noutcome missing values:")
print(outcome.isna().sum())

print("\ndemographics missing values:")
print(demographics.isna().sum())

sessions missing values:
account_id                       0
session_id                       0
session_start                    0
trial_start_date                 0
main_device                      0
genre_watched_per_session        0
search_query_text            56534
session_watch_time               0
total_watch_time                 0
genres_watched_list              0
searches_made                    0
no_results_searches              0
dtype: int64

billing missing values:
account_id                    0
payment_plan_filled_in        0
sign_up_platform              0
region                        0
referral_or_promotion_code    0
trial_plan_chosen             0
dtype: int64

outcome missing values:
account_id                           0
converted_to_paid_subscription       0
date_of_conversion                9885
early_churn                       9885
dtype: int64

demographics missing values:
account_id             0
age_band            1127
stated_interests    1119
dtype: int64


In [27]:
print(outcome.isna().sum())

account_id                           0
converted_to_paid_subscription       0
date_of_conversion                9885
early_churn                       9885
dtype: int64


In [28]:
# check that blank the conversion dates are the non converted users
print(outcome['converted_to_paid_subscription'].value_counts())
print(pd.crosstab(outcome['converted_to_paid_subscription'], outcome['date_of_conversion'].isna()))

converted_to_paid_subscription
False    9885
True     8815
Name: count, dtype: int64
date_of_conversion              False  True 
converted_to_paid_subscription              
False                               0   9885
True                             8815      0


**shows new target counts after the duplicatation removal:** So now 9885 False and 8815 True but the balance is still similar

### In finding missing values 
**What I found:** There are blank search query texts in the "sessions" table, a blank "age_band"and "stated_interests" in the "Demographics" table. The blank conversion dates and early churcn were in the "outcome" table.
**What the blanks mean:** A blank search text indicates there was no search made during the session. The blank conversion dates and early churn values meant the accounts never converted to a paid subscription, which shows in in the True False crosstab. Its only the demographics blanks, that are age band and stated interests are the actual missing value data.
**What I'll do:** Instead of not using and throwing away those accounts, Ill fill in the demographics blanks as "unknown", then with the outcome table, ill drop the conversion date and early churn columns before the modelling process anyway because that can leak the answer. 


In [29]:
# the sample of 10 rows of session start and trial start date to check date formats
print(sessions[['session_start', 'trial_start_date']].sample(10, random_state=1))

           session_start trial_start_date
41057   2025-10-29 20:52       2025-10-22
95653   2025-04-18 05:10       2025-04-08
15663   2024-08-30 03:05       2024-08-28
16520   2026-04-21 23:52       2026-04-14
22996   2024-09-06 08:05       2024-09-04
112794  2025-03-28 16:23       2025-03-27
49630   2026-05-25 17:30       2026-05-11
7751    2025-04-24 01:38       2025-04-10
64259   2025-11-12 16:39       2025-11-04
67290   2026-05-23 18:17       2026-05-22


Training the Model

Validating the Model

Results as Visuals